In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l3.3 Causal masking

A decoder predicts the next token, so a token must never peek at the future.
The causal mask sets future scores to -inf before the softmax, zeroing their
weight.

In [ ]:
from pocketlm import scaled_dot_product_attention

torch.manual_seed(0)
q = torch.randn(1, 4, 8)
k = torch.randn(1, 4, 8)
v = torch.randn(1, 4, 8)
out, w = scaled_dot_product_attention(q, k, v, causal=True)
print("causal weights (upper triangle is 0):")
print(w[0].round(decimals=2))

Proof, not promise: edit every token after the first and the first token's
output does not move. The past cannot see the future.

In [ ]:
k2 = k.clone()
v2 = v.clone()
k2[:, 1:] += 5.0
v2[:, 1:] += 5.0
out2, _ = scaled_dot_product_attention(q, k2, v2, causal=True)
print("position 0 unchanged by future edits:", torch.allclose(out[:, 0], out2[:, 0]))
upper = torch.triu(torch.ones(4, 4), diagonal=1).bool()
assert torch.all(w[0][upper] == 0)
assert torch.allclose(out[:, 0], out2[:, 0], atol=1e-6)